# Обработанный train: 100 строк

`dataset/processed/train.parquet` — результат `src/preprocess.py`. Что он
сделал с исходным `dataset/train.parquet`:

1. удалил вырожденные колонки `search_category` и `search_is_delivery_search`;
2. привёл цену и координаты из `decimal128` в `float64`;
3. разобрал `infm_params_text` модулем `src/infm_params.py` — добавились
   фильтры запроса `filter_*` и поля объявления `item_*`, включая текст для
   ретрива `item_params_text`;
4. удалил разобранные исходные тексты (`search_infm_params_text`,
   `item_infm_params_text`) и заменил `item_category_id` флагом
   `item_is_service`: сам идентификатор равен 114 у 99.997% строк train, но в
   корпусе 1 876 объявлений — товары, а не услуги;
5. поставил колонки в порядке: сначала поля запроса, затем поля объявления.

Из 19 колонок получилось 24.

In [0]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

ROOT = Path("..").resolve() if Path("../src").exists() else Path(".").resolve()
sys.path.insert(0, str(ROOT / "src"))
import infm_params as ip

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 250)
pd.set_option("display.max_colwidth", 60)

TRAIN = ROOT / "dataset" / "processed" / "train.parquet"
TRAIN_RAW = ROOT / "dataset" / "train.parquet"

## 1. Загрузка 100 строк

Читается только первый батч файла, а не все ~400 МБ. Для сравнения «до и
после» рядом читаются те же 100 строк из сырого файла: `preprocess.py`
сохраняет порядок строк.

In [1]:
head = lambda path: next(pq.ParquetFile(path).iter_batches(batch_size=100)).to_pandas()

train = head(TRAIN)
raw = head(TRAIN_RAW)

print("обработанный:", train.shape, "| сырой:", raw.shape)
print("совпадает порядок строк:", bool((train.item_id == raw.item_id).all()))
train.head()

обработанный: (100, 24) | сырой: (100, 19)
совпадает порядок строк: True


,search_query,search_location_id,filter_category,filter_subcategory,filter_subject,filter_online_booking,item_title_raw,item_rating_reviews_count,item_rating,item_price,item_microcat_id,item_longitude,item_location_id,item_latitude,item_is_phone_hidden,item_is_message_forbidden,item_id,item_description_raw,item_is_service,item_category,item_subcategory,item_subjects,item_online_booking,item_params_text
0,скупка телевизоров,652430,NaN,[],[],False,Скупка б/у техники,91.0,4.989011,1.0,2303374,39.712818,652430,54.629230,True,False,e8b685dffe1a408e,Скупаю практически любую современную новую и б/у технику...,True,No category,NaN,[],False,
1,автоподбор,640860,NaN,[],[],False,Автоподбор Разовый осмотр автомобиля,462.0,4.982684,3500.0,2303374,44.051010,640860,56.273390,False,False,92e1b0446f827b59,🚗 Автоподбор и выездная диагностика автомобиля в Нижнем ...,True,No category,NaN,[],False,
2,баня на дровах,653240,"Красота, здоровье","[СПА-услуги, массаж]",[],True,"Баня на дровах ""Прованс"" на Цветочной",NaN,NaN,1600.0,86469,30.128784,653240,59.783688,False,False,624846856ce81d69,"В ритме современной жизни так сложно найти момент, чтоб...",True,"Красота, здоровье","СПА-услуги, массаж",[],True,"Красота, здоровье; СПА-услуги, массаж; Спа-процедуры; Ар..."
3,изготовление госномера на авто,634670,"Оборудование, производство",[],[],False,"Изготовление дубликатов авто номеров, гос номеров",14.0,4.714286,1700.0,2303428,40.537841,633570,45.424875,False,False,45b8628b9c6e7b85,Изготовим дубликат номера по утере или износу на официал...,True,"Оборудование, производство","Производство, обработка",[],False,"Оборудование, производство; Производство, обработка"
4,укладка плитки,658430,Ремонт и отделка,[Ремонт квартир и домов под ключ],[],False,Ремонт и отделка квартир под ключ,1.0,5.000000,1000.0,44725,69.496445,658430,56.105984,False,False,c95a4a7daf2a967f,отделочные работы любой сложности.,True,Ремонт и отделка,Ремонт квартир и домов под ключ,[],False,Ремонт и отделка; Ремонт квартир и домов под ключ; Все в...


## 2. Что изменилось в составе колонок

In [2]:
FILTER_COLS = ["filter_category", "filter_subcategory", "filter_subject",
               "filter_online_booking"]
ITEM_COLS = ["item_category", "item_subcategory", "item_subjects",
             "item_online_booking", "item_params_text"]

pd.DataFrame({
    "колонки": [", ".join(sorted(set(raw.columns) - set(train.columns))),
                ", ".join(["item_is_service"] + FILTER_COLS + ITEM_COLS)],
}, index=["удалены", "добавлены"])

,колонки
удалены,"item_category_id, item_infm_params_text, search_category..."
добавлены,"item_is_service, filter_category, filter_subcategory, fi..."


## 3. Фильтры запроса: исходный текст и результат разбора

Исходный текст берём из сырого файла — в обработанном его уже нет.

In [3]:
with pd.option_context("display.max_colwidth", 90):
    display(pd.concat([train[["search_query"]],
                       raw[["search_infm_params_text"]],
                       train[FILTER_COLS]], axis=1).head(15))

,search_query,search_infm_params_text,filter_category,filter_subcategory,filter_subject,filter_online_booking
0,скупка телевизоров,,NaN,[],[],False
1,автоподбор,Рейтинг пользователя 4 звезды и выше,NaN,[],[],False
2,баня на дровах,"Онлайн-запись Тип услуги СПА-услуги, массаж Вид услуги Красота, здоровье","Красота, здоровье","[СПА-услуги, массаж]",[],True
3,изготовление госномера на авто,"Вид услуги Оборудование, производство","Оборудование, производство",[],[],False
4,укладка плитки,Тип услуги Ремонт квартир и домов под ключ Вид услуги Ремонт и отделка,Ремонт и отделка,[Ремонт квартир и домов под ключ],[],False
5,сборка мебели,Тип услуги Сборка и ремонт мебели Вид услуги Ремонт и отделка,Ремонт и отделка,[Сборка и ремонт мебели],[],False
6,наращивание ногтей,"Онлайн-запись Тип услуги Маникюр, педикюр Вид услуги Красота, здоровье","Красота, здоровье","[Маникюр, педикюр]",[],True
7,установка входных дверей,,NaN,[],[],False
8,выкуп компьютеров,Вид услуги,No category,[],[],False
9,массаж,Где вы оказываете услуги У себя дома Ваши клиенты Мужчины Кто оказывает услуги Женщина...,"Красота, здоровье","[СПА-услуги, массаж]",[],False


## 4. Поля объявления: исходный текст и результат разбора

In [4]:
with pd.option_context("display.max_colwidth", 70):
    display(pd.concat([train[["item_title_raw"]],
                       raw[["item_infm_params_text"]],
                       train[ITEM_COLS]], axis=1).head(10))

,item_title_raw,item_infm_params_text,item_category,item_subcategory,item_subjects,item_online_booking,item_params_text
0,Скупка б/у техники,"Вид услуги Место оказания услуг Первомайский пр-т, 66 Тип стоимост...",No category,NaN,[],False,
1,Автоподбор Разовый осмотр автомобиля,"Вид услуги Место оказания услуг Нижний Новгород, Советский район, ...",No category,NaN,[],False,
2,"Баня на дровах ""Прованс"" на Цветочной","Вид услуги Красота, здоровье Место оказания услуг Санкт-Петербург,...","Красота, здоровье","СПА-услуги, массаж",[],True,"Красота, здоровье; СПА-услуги, массаж; Спа-процедуры; Аренда бани;..."
3,"Изготовление дубликатов авто номеров, гос номеров","Вид услуги Оборудование, производство Тип услуги Производство, обр...","Оборудование, производство","Производство, обработка",[],False,"Оборудование, производство; Производство, обработка"
4,Ремонт и отделка квартир под ключ,Вид услуги Ремонт и отделка Тип услуги Ремонт квартир и домов под ...,Ремонт и отделка,Ремонт квартир и домов под ключ,[],False,Ремонт и отделка; Ремонт квартир и домов под ключ; Все виды отделк...
5,Сборка мебели,Вид услуги Ремонт и отделка Тип услуги Сборка и ремонт мебели Мест...,Ремонт и отделка,Сборка и ремонт мебели,[],False,Ремонт и отделка; Сборка и ремонт мебели; Ремонт шкафов; Реставрац...
6,Мастер маникюр и педикюра,"Вид услуги Красота, здоровье Место оказания услуг Москва, Новокоси...","Красота, здоровье","Маникюр, педикюр",[],True,"Красота, здоровье; Маникюр, педикюр; Аппаратный маникюр; Комбиниро..."
7,Установка межкомнатных дверей,Вид услуги Ремонт и отделка Тип услуги Двери Место оказания услуг ...,Ремонт и отделка,Двери,[],False,Ремонт и отделка; Двери; Установка межкомнатных дверей; Установка ...
8,"Выкуп пк, ноутбуков,приставок","Вид услуги Место оказания услуг ул. Братьев Кашириных, 131А Тип ст...",No category,NaN,[],False,
9,Массаж,"Вид услуги Красота, здоровье Место оказания услуг ул. Лейтенанта Ш...","Красота, здоровье","СПА-услуги, массаж",[],False,"Красота, здоровье; СПА-услуги, массаж; Массаж; Классический массаж..."


## 5. Одна строка целиком

Колонки идут в том порядке, в каком лежат в файле: сначала поля запроса
(`search_*`, `filter_*`), затем поля объявления.

In [5]:
with pd.option_context("display.max_colwidth", 300):
    display(train.iloc[[1]].T.rename(columns={1: "значение"}))

,значение
search_query,автоподбор
search_location_id,640860
filter_category,NaN
filter_subcategory,[]
filter_subject,[]
filter_online_booking,False
item_title_raw,Автоподбор Разовый осмотр автомобиля
item_rating_reviews_count,462.0
item_rating,4.982684
item_price,3500.0


## 6. Выполняет ли объявление фильтры запроса

`match_filters`: `True` — прошёл, `False` — не прошёл, `None` — фильтр не
задан.

In [6]:
checks = pd.DataFrame(
    [ip.match_filters(f, i) for f, i in zip(train[FILTER_COLS].to_dict("records"),
                                            train[ITEM_COLS].to_dict("records"))],
    index=train.index,
)
display(pd.concat([train[["search_query", "filter_category", "item_category"]], checks],
                  axis=1).head(15))

pd.DataFrame({
    "фильтр задан, строк": checks.notna().sum(),
    "объявление прошло": checks.apply(lambda s: s.dropna().astype(bool).mean()),
}).style.format({"объявление прошло": "{:.0%}"}, na_rep="—")

,search_query,filter_category,item_category,category,subcategory,subject,online_booking
0,скупка телевизоров,NaN,No category,None,None,None,None
1,автоподбор,NaN,No category,None,None,None,None
2,баня на дровах,"Красота, здоровье","Красота, здоровье",True,True,None,True
3,изготовление госномера на авто,"Оборудование, производство","Оборудование, производство",True,None,None,None
4,укладка плитки,Ремонт и отделка,Ремонт и отделка,True,True,None,None
5,сборка мебели,Ремонт и отделка,Ремонт и отделка,True,True,None,None
6,наращивание ногтей,"Красота, здоровье","Красота, здоровье",True,True,None,True
7,установка входных дверей,NaN,Ремонт и отделка,None,None,None,None
8,выкуп компьютеров,No category,No category,True,None,None,None
9,массаж,"Красота, здоровье","Красота, здоровье",True,True,None,None


,"фильтр задан, строк",объявление прошло
category,65,100%
subcategory,36,100%
subject,1,100%
online_booking,4,100%


## 7. Типы и заполненность колонок

Заполнено — не `None`/`NaN`, не пустая строка и не пустой список. Метки
«No category» / «No subcategory» и `False` считаются заполненными значениями.

In [7]:
def filled(series):
    def has_value(x):
        if isinstance(x, (list, tuple, np.ndarray)):
            return len(x) > 0
        return pd.notna(x) and x != ""
    return series.map(has_value).mean()


pd.DataFrame({
    "тип": train.dtypes.astype(str),
    "заполнено": [filled(train[c]) for c in train.columns],
}).style.format({"заполнено": "{:.0%}"})

,тип,заполнено
search_query,str,100%
search_location_id,int64,100%
filter_category,str,65%
filter_subcategory,object,36%
filter_subject,object,1%
filter_online_booking,bool,100%
item_title_raw,str,100%
item_rating_reviews_count,float64,99%
item_rating,float64,98%
item_price,float64,100%
